# Prototipo 1.8 — Priorización de riesgo con validación temporal

Reescritura de la versión 1.6 a partir de mediciones, no de intuiciones. Cada decisión de
diseño de abajo se tomó comparando alternativas sobre datos reales, y varias contradicen lo
que parecía razonable.

| Decisión | Alternativa descartada | Evidencia |
|---|---|---|
| **LightGBM solo** | Stacking XGB + LGBM | AUC 0.7765 vs 0.7381, y 9× más rápido |
| **Ventana de 1 periodo** | Historia completa (11 años) | AUC 0.7381 vs 0.7253 con 13,6 % de los datos |
| **Operar por capacidad** | Umbral de probabilidad | El umbral no se transfiere entre periodos |
| **Partición temporal** | Partición aleatoria | La aleatoria sobrestima el AUC en 0.101 |
| **Preprocesamiento en el pipeline** | `fit` antes del split | Evita que el test influya en el entrenamiento |
| **15 predictores** | 17 | Dos eran constantes tras la imputación |

## Por qué cambió el enfoque

La tasa de deserción **sube de forma sostenida entre periodos**: 9,8 % → 11,8 % → 15,1 % en
tres años. Eso es deriva de concepto, y reencuadra el problema: no es un modelo con fugas que
haya que limpiar, es un modelo que **envejece rápido**. El diseño lo asume en lugar de
combatirlo — reentrena con datos recientes y se opera por ranking, no por probabilidad
absoluta.

## Qué se descartó midiendo

Tres sospechas razonables que **no** resultaron ciertas, documentadas para que nadie las
vuelva a perseguir:

- **Fuga por `TIPO_SALTO`**: AUC 0.5104 usándola sola. No filtra el target.
- **Fuga por sujetos repetidos**: eliminar todo el solape entre train y test mueve el AUC
  0.000. Las variables describen el periodo, no a la persona.
- **Recalibrar el umbral cada periodo**: recupera 85,8 % del máximo alcanzable, contra
  85,5 % sin recalibrar. La diferencia es ruido.

## 1. Dependencias

In [ ]:
import os
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib
from sqlalchemy import create_engine, text

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import roc_auc_score, precision_recall_curve
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier

import shap
import matplotlib.pyplot as plt

## 2. Conexión

Host, puerto y nombre de la base se leen del entorno. Copia `.env.example` a `.env` y
complétalo; nunca escribas credenciales en el notebook.

In [ ]:
host     = os.getenv('DB_HOST')
port     = os.getenv('DB_PORT', '1433')
database = os.getenv('DB_NAME')

if not all([host, database]):
    raise RuntimeError("Faltan variables de entorno: define DB_HOST y DB_NAME. Ver .env.example")

url = ('mssql+pyodbc://@{host}:{port}/{db}'
       '?trusted_connection=yes&driver=SQL+Server').format(host=host, port=port, db=database)
engine = create_engine(url)
print("Conexion establecida")

## 3. Extracción

Seis fuentes que se integran por la llave compuesta `Identificacion` + `Periodo`. Los nombres
de esquema y tabla son genéricos; sustitúyelos por los de tu instalación.

In [ ]:
CONSULTAS = {
    'historial': '''
        SELECT Identificacion, Periodo, Status, Tipo_Salto, Modalidad,
               Semestre_SINU, año, Genero, Estado_Pago, RANGO_EDAD, RANGO_SALARIO,
               ESTA_TRABAJANDO, METODO_FINANCIAMIENTO, ZONA_RESIDENCIA,
               REGIMEN_SISTEMA_SALUD
        FROM academico.historial_academico
    ''',
    'materias': '''
        SELECT IDENTIFICACION AS Identificacion, COD_PERIODO AS Periodo,
               [MATERIAS INSCRITAS], [MATERIAS APROBADAS], Porcentaje_aprobacion
        FROM academico.aprobacion_materias
    ''',
    'cartera': '''
        SELECT Identificacion, Periodo, TOTAL
        FROM financiera.cartera
    ''',
}

datos = {}
with engine.connect() as conn:
    for nombre, sql in CONSULTAS.items():
        datos[nombre] = pd.read_sql_query(text(sql), conn)
        print(f"  {nombre:<12} {datos[nombre].shape}")

## 4. Integración

Cada fuente se deduplica **antes** del merge. La versión anterior no lo hacía y uno de los
`LEFT JOIN` inflaba silenciosamente el número de filas por llaves repetidas en origen.

In [ ]:
df = datos['historial'].copy()
filas_iniciales = len(df)

for nombre in ['materias', 'cartera']:
    aux = datos[nombre].drop_duplicates(subset=['Identificacion', 'Periodo'], keep='last')
    df = df.merge(aux, on=['Identificacion', 'Periodo'], how='left')

assert len(df) == filas_iniciales, (
    f"El merge cambio el numero de filas: {filas_iniciales} -> {len(df)}. "
    "Revisa la deduplicacion de las fuentes."
)

df.columns = df.columns.str.strip().str.upper().str.replace(' ', '_')
print(f"Integrado: {df.shape}")

## 5. Target e ingeniería de características

`FLAG_NO_APROBO_NADA` separa "aprobó 0 %" de "no hay dato". Sin esa distinción la imputación
por mediana colapsa la señal de riesgo más fuerte del dataset contra un simple faltante.

In [ ]:
# El valor que marca la deserción en el campo de estado se lee del entorno:
# es específico de cada instalación y no debe quedar escrito aquí.
VALOR_DESERCION = os.getenv('STATUS_DESERCION', '').strip().lower()
if not VALOR_DESERCION:
    raise RuntimeError("Define STATUS_DESERCION en el entorno. Ver .env.example")

df['TARGET_DESERCION'] = (
    df['STATUS'].astype(str).str.strip().str.lower() == VALOR_DESERCION
).astype(int)

if df['TARGET_DESERCION'].sum() == 0:
    raise RuntimeError(
        f"Ninguna fila coincide con STATUS_DESERCION={VALOR_DESERCION!r}. "
        f"Valores presentes: {df['STATUS'].astype(str).str.strip().str.lower().unique()[:10]}"
    )

df['FLAG_NO_APROBO_NADA'] = np.where(df['PORCENTAJE_APROBACION'] == 0, 1, 0)
df['PORCENTAJE_APROBACION'] = df['PORCENTAJE_APROBACION'].replace(0, np.nan)

print(df['TARGET_DESERCION'].value_counts(normalize=True).round(4).to_dict())

## 6. Selección de variables

`ESTADO_PAGO` resultó **constante** en todo el dataset y `ASISTENCIA` concentraba el 99,69 %
de las filas en un mismo valor tras la imputación: llegaba con más del 90 % de faltantes y la
moda terminó de aplanarla. Ninguna de las dos aporta información, así que se eliminan.

In [ ]:
NUMERICAS = [
    'SEMESTRE_SINU', 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS',
    'PORCENTAJE_APROBACION', 'TOTAL', 'FLAG_NO_APROBO_NADA',
]
CATEGORICAS = [
    'TIPO_SALTO', 'MODALIDAD', 'GENERO', 'RANGO_EDAD', 'RANGO_SALARIO',
    'ESTA_TRABAJANDO', 'METODO_FINANCIAMIENTO', 'ZONA_RESIDENCIA',
    'REGIMEN_SISTEMA_SALUD',
]
# Descartadas por medición: ESTADO_PAGO (constante), ASISTENCIA (99,69 % un valor)

FEATURES = NUMERICAS + CATEGORICAS
TARGET = 'TARGET_DESERCION'
print(f"{len(FEATURES)} predictores declarados")

### Control de cobertura

Una variable que llega completamente vacía no produce ningún error: `SimpleImputer` la
descarta en silencio y el modelo entrena con menos predictores de los declarados, sin aviso.
Ocurrió en este proyecto —una tabla de origen quedó vacía y su variable desapareció sin que
nada fallara— así que la cobertura se verifica de forma explícita.

In [ ]:
COBERTURA_MINIMA = 0.05   # por debajo de esto la variable no aporta señal utilizable

cobertura = df[FEATURES].notna().mean().sort_values()
descartadas = cobertura[cobertura < COBERTURA_MINIMA].index.tolist()

print("Cobertura por variable:")
for v, c in cobertura.items():
    marca = "  <-- DESCARTADA" if v in descartadas else ""
    print(f"  {v:<26} {c*100:5.1f} %{marca}")

if descartadas:
    print(f"\nSe descartan {len(descartadas)} variable(s) sin cobertura suficiente: {descartadas}")
    NUMERICAS = [c for c in NUMERICAS if c not in descartadas]
    CATEGORICAS = [c for c in CATEGORICAS if c not in descartadas]
    FEATURES = NUMERICAS + CATEGORICAS

print(f"\n{len(FEATURES)} predictores efectivos")

## 7. Partición temporal

Entrenar con el periodo más reciente disponible y evaluar sobre el siguiente. Reproduce la
situación real: predecir un futuro que el modelo no vio.

La partición aleatoria de la versión anterior sobrestimaba el AUC en **0.101** al mezclar
periodos. La ventana de un año no es una limitación de datos sino el resultado de medir: con
11 años de historia el AUC baja a 0.7253, con uno sube a 0.7381.

In [ ]:
df['AÑO'] = df['AÑO'].astype(int)
periodos = sorted(df['AÑO'].unique())
anio_test = periodos[-1]
anio_train = periodos[-2]

train = df[df['AÑO'] == anio_train]
test = df[df['AÑO'] == anio_test]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {anio_train}: {len(train):>8,} filas | tasa {y_train.mean():.4f}")
print(f"test  {anio_test}: {len(test):>8,} filas | tasa {y_test.mean():.4f}")
print(f"\nDeriva de la tasa base: {(y_test.mean()/y_train.mean()-1)*100:+.1f} %")

## 8. Pipeline

Todo el preprocesamiento vive **dentro** del pipeline. La versión anterior ajustaba el
`StandardScaler` y los imputadores sobre el dataset completo antes de partir, dejando que las
estadísticas del conjunto de prueba influyeran en el entrenamiento.

Se usa `scale_pos_weight` en lugar de SMOTE: consigue el mismo efecto de balanceo sin
sintetizar filas, y evita el riesgo de generar vecinos artificiales a través de variables
categóricas codificadas como enteros, donde la interpolación no tiene sentido.

In [ ]:
preprocesador = ColumnTransformer([
    ('num', ImbPipeline([
        ('imputar', SimpleImputer(strategy='median')),
        ('escalar', StandardScaler()),
    ]), NUMERICAS),
    ('cat', ImbPipeline([
        ('imputar', SimpleImputer(strategy='most_frequent')),
        ('codificar', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), CATEGORICAS),
], remainder='drop')

peso = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

modelo = ImbPipeline([
    ('preprocesador', preprocesador),
    ('clasificador', LGBMClassifier(
        n_estimators=481, num_leaves=32, learning_rate=0.0244, subsample=0.9122,
        random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight=peso,
    )),
])

modelo.fit(X_train, y_train)
print("Entrenado")

## 9. Evaluación por capacidad

**El cambio conceptual de esta versión.** El umbral de probabilidad no se transfiere entre
periodos: el óptimo pasa de 0.26 a 0.01 de un año al siguiente, porque la deriva de la tasa
base descalibra las probabilidades. El *ranking*, en cambio, se mantiene.

Así que el sistema no responde "¿deserta este estudiante?" sino **"¿cuáles son los N de mayor
riesgo?"**, donde N es lo que el equipo de retención puede atender. Es la pregunta que
realmente se hace un programa de permanencia, y es inmune a la descalibración.

In [ ]:
proba = modelo.predict_proba(X_test)[:, 1]
tasa_base = y_test.mean()
orden = np.argsort(-proba)
verdad = y_test.to_numpy()

print(f"AUC-ROC: {roc_auc_score(y_test, proba):.4f}")
print(f"Tasa base (alertar al azar): {tasa_base:.4f}\n")

filas = []
for pct in [1, 2, 5, 10, 15, 20, 30]:
    k = int(len(verdad) * pct / 100)
    sel = orden[:k]
    precision = verdad[sel].mean()
    filas.append({
        'top_%': pct, 'alertas': k, 'aciertos': int(verdad[sel].sum()),
        'precision': round(precision, 4), 'lift': round(precision / tasa_base, 2),
        'recall': round(verdad[sel].sum() / verdad.sum(), 4),
    })
pd.DataFrame(filas)

### Cómo se lee esta tabla

`lift` es cuántas veces mejor que elegir al azar. Un lift de 6 en el top 1 % significa que de
cada 100 estudiantes priorizados, 6 veces más desertores que si se eligiera sin modelo.

La decisión operativa es **elegir la fila según la capacidad real del equipo**, no maximizar
una métrica. Un programa que puede acompañar a 1.000 estudiantes por periodo opera en el
top 1 %, donde la precisión es muy alta y el desperdicio de recursos mínimo. Uno que puede
llegar a 10.000 acepta menos precisión a cambio de cubrir más casos.

## 10. Interpretabilidad

In [ ]:
X_test_proc = modelo.named_steps['preprocesador'].transform(X_test)
explicador = shap.TreeExplainer(modelo.named_steps['clasificador'])
shap_values = explicador.shap_values(X_test_proc)

shap.summary_plot(shap_values, X_test_proc, feature_names=FEATURES,
                  plot_type='bar', max_display=15, show=False)
plt.title("Importancia global de las variables")
plt.tight_layout()
plt.show()

### Explicación individual

Cada alerta debe poder justificarse ante el comité que va a actuar sobre ella. Sin esto el
sistema pide intervenir sobre una persona sin decir por qué.

In [ ]:
def explicar(posicion_en_ranking=0):
    i = orden[posicion_en_ranking]
    return shap.force_plot(
        explicador.expected_value, shap_values[i], X_test_proc[i],
        feature_names=FEATURES, matplotlib=True,
    )

explicar(0)   # el estudiante de mayor riesgo

## 11. Persistencia

Se guarda el pipeline junto a los metadatos que hacen falta para saber **cuándo caduca**. Un
modelo entrenado sobre un periodo concreto, en un fenómeno con deriva medida, no es válido
indefinidamente.

In [ ]:
metadatos = {
    'version': '1.8',
    'entrenado_utc': datetime.now(timezone.utc).isoformat(),
    'periodo_entrenamiento': int(anio_train),
    'periodo_evaluacion': int(anio_test),
    'tasa_base_entrenamiento': float(y_train.mean()),
    'tasa_base_evaluacion': float(y_test.mean()),
    'auc': float(roc_auc_score(y_test, proba)),
    'features': FEATURES,
    'modo_operacion': 'ranking por capacidad; NO usar umbral fijo de probabilidad',
    'caducidad': 'reentrenar cada periodo academico: la tasa base deriva ~+25 % anual',
}

modelo.metadatos = metadatos
joblib.dump(modelo, 'modelo_18.pkl')
with open('modelo_18_metadatos.json', 'w', encoding='utf-8') as fh:
    json.dump(metadatos, fh, indent=2, ensure_ascii=False)

print(json.dumps(metadatos, indent=2, ensure_ascii=False)[:400])

## 12. Scoring

La función devuelve el ranking completo con su posición, y marca los `capacidad` primeros. No
expone un umbral de probabilidad, deliberadamente: sería la vía más fácil de reintroducir el
error que esta versión corrige.

In [ ]:
def priorizar(df_nuevos, capacidad, modelo_path='modelo_18.pkl'):
    """Ordena estudiantes por riesgo y marca los `capacidad` de mayor prioridad.

    :param df_nuevos: DataFrame con las columnas de FEATURES.
    :param capacidad: cuántos estudiantes puede atender el programa este periodo.
    :return: DataFrame ordenado por riesgo descendente.
    """
    pipeline = joblib.load(modelo_path)
    meta = getattr(pipeline, 'metadatos', {})

    faltantes = set(meta.get('features', [])) - set(df_nuevos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas por el modelo: {sorted(faltantes)}")

    p = pipeline.predict_proba(df_nuevos[meta['features']])[:, 1]
    salida = df_nuevos[['IDENTIFICACION', 'PERIODO']].copy()
    salida['RIESGO'] = p
    salida = salida.sort_values('RIESGO', ascending=False).reset_index(drop=True)
    salida['POSICION'] = np.arange(1, len(salida) + 1)
    salida['PRIORIZADO'] = salida['POSICION'] <= capacidad

    print(f"Modelo entrenado sobre el periodo {meta.get('periodo_entrenamiento')}. "
          f"{meta.get('caducidad', '')}")
    return salida


# priorizar(df_periodo_actual, capacidad=1000)